# Source Dimension Loader

This notebook maintains the `warehouse.dim_source` dimension table.

**Purpose**: Track data source systems and their metadata

**Key Features**:
* Stable surrogate keys for source systems
* Source metadata and configuration
* SCD Type 1 (overwrite on change)

**Source**: Derived from distinct sources in `silver.silver_jobs_current`

**Target**: `workspace.warehouse.dim_source`

In [0]:
%sql
-- Extract distinct sources from silver layer
CREATE OR REPLACE TEMP VIEW source_extract AS
SELECT DISTINCT
  source_name,
  COUNT(*) OVER (PARTITION BY source_name) as job_count,
  MIN(created_at) OVER (PARTITION BY source_name) as first_seen,
  MAX(updated_at) OVER (PARTITION BY source_name) as last_seen
FROM workspace.silver.silver_jobs_current
WHERE source_name IS NOT NULL;

In [0]:
%sql
-- Generate or update source dimension with stable keys
CREATE OR REPLACE TEMP VIEW source_with_keys AS
SELECT
  COALESCE(d.source_sk, ROW_NUMBER() OVER (ORDER BY s.source_name) + COALESCE(max_sk, 0)) as source_sk,
  s.source_name,
  COALESCE(s.source_name, 'UNKNOWN') as source_display_name,
  CASE s.source_name
    WHEN 'linkedin' THEN 'https://www.linkedin.com/jobs'
    WHEN 'indeed' THEN 'https://www.indeed.com'
    WHEN 'glassdoor' THEN 'https://www.glassdoor.com'
    ELSE NULL
  END as source_url,
  CASE s.source_name
    WHEN 'linkedin' THEN 'API'
    WHEN 'indeed' THEN 'API'
    WHEN 'glassdoor' THEN 'SCRAPE'
    ELSE 'UNKNOWN'
  END as ingestion_method,
  TRUE as is_active,
  s.first_seen,
  s.last_seen,
  CURRENT_TIMESTAMP() as updated_at
FROM source_extract s
LEFT JOIN workspace.warehouse.dim_source d
  ON s.source_name = d.source_name
CROSS JOIN (
  SELECT COALESCE(MAX(source_sk), 0) as max_sk 
  FROM workspace.warehouse.dim_source
) max_keys;

In [0]:
%sql
-- Merge into target dimension (SCD Type 1)
MERGE INTO workspace.warehouse.dim_source target
USING source_with_keys source
ON target.source_name = source.source_name
WHEN MATCHED THEN UPDATE SET
  target.source_display_name = source.source_display_name,
  target.source_url = source.source_url,
  target.ingestion_method = source.ingestion_method,
  target.is_active = source.is_active,
  target.last_seen = source.last_seen,
  target.updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT (
  source_sk,
  source_name,
  source_display_name,
  source_url,
  ingestion_method,
  is_active,
  first_seen,
  last_seen,
  updated_at
) VALUES (
  source.source_sk,
  source.source_name,
  source.source_display_name,
  source.source_url,
  source.ingestion_method,
  source.is_active,
  source.first_seen,
  source.last_seen,
  source.updated_at
);

In [0]:
%sql
-- Validate source dimension
SELECT 
  source_sk,
  source_name,
  source_display_name,
  ingestion_method,
  is_active,
  first_seen,
  last_seen
FROM workspace.warehouse.dim_source
ORDER BY source_sk;